# CS 3120/5120: Secure Distributed Computation
## In-Class Exercise, Week of 1/12/2026

# Question 1

Write functions `share` and `reconstruct` for an additive secret sharing scheme over the integers. The functions should assume two secret shares.

In [81]:
import numpy as np

def share(x):
    # Step 1: create share 1 as a random number
    s1 = np.random.randint(0, 10000000)
    s2 = x - s1
    return s1, s2

def reconstruct(pair):
    s1, s2 = pair
    return s1 + s2

In [82]:
share(10)[1]

-4722524

In [85]:
# was the original secret 100 million or 10?
# second share actually reveals the answer!
# security is broken
share(100000000)[1]

96340552

In [15]:
share(12)

(46, -34)

In [7]:
# Test case
assert reconstruct(share(5)) == 5
assert reconstruct(share(0)) == 0

a1, a2 = share(5)
assert reconstruct((a1 + 10, a2)) == 15

b1, b2 = share(20)
assert reconstruct((a1 + b1, a2 + b2)) == 25

In [35]:
x1, x2 = share(3)
y1, y2 = share(5)
sum_s1 = x1 + y1
sum_s2 = x2 + y2
reconstruct((sum_s1, sum_s2)) # result should be x + y

8

# Question 2

Is the above secret sharing scheme *additively homomorphic*? Why or why not?

Is it *multiplicatively homomorphic*? Why or why not?

1. Yes. We can add shares of two secrets to get new shares of the sum of the secrets. Adding shares == sharing the sum of the secrets.
2. No. If you multiply shares of the original secrets, you don't get shares of the product of secrets.

# Question 3

What is problematic (for security) about the use of integers in the above solution?

- Basic issue: it's not really possible to "sample a random integer"
- In cryptography, we always sample randomness from a finite set to solve this problem. The most common case is bits: {0, 1}
- In this class, we often use larger finite sets like finite fields (Galois fields)
- $GF(p)$ = {0, ..., p-1} where $p$ is prime
- To fix our secret sharing scheme, we will require that secret shares are in $GF(p)$ and we will use modular arithmetic everywhere
- $GF(2)$ is exactly the bits
- Other common primes: 2^31 - 1, 2^61 -1

## Question 4

Write functions `plusFE` and `multFE` that add and multiply two field elements in $GF(p)$, respectively.

In [90]:
(3+96) %  p

2

In [172]:
p = 97
# represent the integer 10 as the field element 10

def plusFE(a, b):
    return (a + b) % p
    
def multFE(a, b):
    return (a * b) % p

In [92]:
# Test case
assert plusFE(2, 3) == 5
assert plusFE(50, 50) == 3
assert multFE(2, 5) == 10
assert multFE(6, 20) == 23

## Question 5

Write functions `share` and `reconstruct` for an additive secret sharing scheme over $GF(p)$. The functions should assume two secret shares.

In [95]:
def share(x):
    s1 = np.random.randint(0, p)
    s2 = plusFE(x, -s1)
    return s1, s2

def reconstruct(shares):
    s1, s2 = shares
    return plusFE(s1, s2)

In [116]:
x1, x2 = share(5)
(x1 + x2) % p

5

In [117]:
# Test case
assert reconstruct(share(5)) == 5
assert reconstruct(share(0)) == 0

a1, a2 = share(5)
assert reconstruct((a1 + 10, a2)) == 15

b1, b2 = share(20)
assert reconstruct((a1 + b1, a2 + b2)) == 25

# Question 6

Write a protocol for two parties to sum their inputs. Each party provides a single input as an integer, and the parties send their numbers to each other so that both can calculate the result.

In [146]:
import pychor

p1 = pychor.Party('p1')
p2 = pychor.Party('p2')

# To compute on located values, use local functions
# Input to a local function might be a located value, and we can do regular computation on it inside the fn
# But the output of the function will be a located value known only to the people who knew the input
@pychor.local_function
def located_plus(x, y):
    return x + y

def protocol_sum_int(v1, v2):
    # P1 sends v1 to P2
    v1.send(p1, p2)

    # P2 sends v2 to P1
    v2.send(p2, p1)
    print("abs output", located_abs(v2))

    # Both parties add up the values to get the total
    #return v1 + v2
    return located_plus(v1, v2)

# with pychor.LocalBackend():
#     # create a located value
#     x = p1.constant(1)
#     # send method on located values allows sending them
#     x.send(p1, p2)
#     print(x, type(x))

In [147]:
with pychor.LocalBackend():
    # Define inputs for the protocol, located at the two parties
    v1 = p1.constant(5)
    v2 = p2.constant(3)
    
    # Run the protocol
    result = protocol_sum_int(v1, v2)

    # Check the results
    assert result.val == 8
    assert result.parties == {p1, p2}
    print(result, v1, v2)

inside located abs: 3 <class 'int'>
abs output 3@{p2, p1}
8@{p2, p1} 5@{p2, p1} 3@{p2, p1}


In [130]:
# Test case
with pychor.LocalBackend():
    # Define inputs for the protocol, located at the two parties
    v1 = p1.constant(5)
    v2 = p2.constant(3)
    
    # Run the protocol
    result = protocol_sum_int(v1, v2)

    # Check the results
    assert result.val == 8
    assert result.parties == {p1, p2}

# Question 7

Write a protocol for two parties to sum their inputs, but with secret sharing. Each party provides a single input as an integer. Each party sends *one secret share* of their number to the other. The parties sum the shares they hold, broadcast the results, and reconstruct the sum from the final shares.

In [163]:
x = (1, 2, 3)
a, b, c = x
c

3

In [200]:
@pychor.local_function
def share(x):
    s1 = np.random.randint(0, p)
    s2 = plusFE(x, -s1)
    return s1, s2

@pychor.local_function
def reconstruct(shares):
    s1, s2 = shares
    return plusFE(s1, s2)

def protocol_sum_secure(v1, v2):
    # Step 1: each party splits their input into secret shares
    # untup method converts a located tuple into a tuple of located  values
    v11, v12 = share(v1).untup(2)
    v21, v22 = share(v2).untup(2)

    # Step 2: each party sends one share to the other party
    v12.send(p1, p2)
    v21.send(p2, p1)

    # Step 3: each party adds the shares they have to get one share of the total
    sum1 = v11 + v21
    sum2 = v12 + v22

    # Step 4: each party sends their share of the total to the other party
    sum1.send(p1, p2)
    sum2.send(p2, p1)

    # Step 5: the parties add the shares to get the total
    return reconstruct([sum1, sum2])

In [201]:
with pychor.LocalBackend():
    # Define inputs for the protocol, located at the two parties
    v1 = p1.constant(5)
    v2 = p2.constant(3)
    
    # Run the protocol
    result = protocol_sum_secure(v1, v2)
    print(result)

    # Check the results
    assert result.val == 8
    assert result.parties == {p1, p2}

49@{p1} 53@{p1} 89@{p2} 11@{p2}
8@{p2, p1}


In [202]:
# Test case
with pychor.LocalBackend():
    # Define inputs for the protocol, located at the two parties
    v1 = p1.constant(5)
    v2 = p2.constant(3)
    
    # Run the protocol
    result = protocol_sum_secure(v1, v2)

    # Check the results
    assert result.val == 8
    assert result.parties == {p1, p2}

11@{p1} 91@{p1} 67@{p2} 33@{p2}


# Question 8

Is `protocol_sum_secure` really secure? In what contexts? For what definition of "secure?"

YOUR ANSWER HERE